# 第4讲：从样本认识总体

## 置信区间与假设检验

贯穿案例：根据25次装配式墙板安装记录，估计总体平均安装用时，并与50分钟计划基准和“平均节省4分钟”工程关注线比较。

## 0. 学习目标与运行顺序

- 区分样本标准差与样本均值标准误；
- 观察样本均值的抽样波动；
- 计算总体均值的95% t置信区间；
- 完成单样本双侧t检验；
- 按“差值—区间—p值—工程阈值”解释结果。

运行顺序：环境 → 数据 → 描述 → 抽样 → 区间与检验 → 工程解释。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def locate_sample():
    candidates = [
        Path('data') / 'DS-L04_墙板安装用时样本.csv',
        Path('../student_release/data') / 'DS-L04_墙板安装用时样本.csv',
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError('请保持Notebook与data目录的相对位置。')

sample_path = locate_sample()
print('Python:', sys.version.split()[0])
print('样本文件:', sample_path.as_posix())

## 1. 样本记录与统计对象

- 一行代表一次装配式墙板安装事件；
- 分析变量为`installation_time_min`，单位为分钟；
- 样本量为25；
- 总体参数为同类安装事件的平均用时`μ`。

In [ ]:
df = pd.read_csv(sample_path)
x = df['installation_time_min'].to_numpy()
n = len(x)
mean = x.mean()
sd = x.std(ddof=1)
summary = pd.DataFrame({'n':[n], 'mean_min':[mean], 'sample_sd_min':[sd]})
display(df.head())
summary.round(6)

样本均值描述25次安装的中心水平；样本标准差描述单次安装围绕均值的离散程度。主案例预期得到`mean=47`、`sample_sd=5`。

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.hist(x, bins=8, color='#277DA1', edgecolor='white')
ax.axvline(mean, color='#E76F51', lw=2, label=f'样本均值={mean:.2f}')
ax.axvline(50, color='#263238', lw=2, ls='--', label='计划基准=50')
ax.set_xlabel('安装用时（分钟）')
ax.set_ylabel('记录数')
ax.legend(frameon=False)
ax.set_title('DS-L04样本分布')
plt.show()

## 2. 重复抽样与样本均值的抽样分布

下方的教学总体只用于观察统计量怎样随抽样变化。每次抽样产生一个样本均值；许多样本均值形成抽样分布。

In [ ]:
rng = np.random.default_rng(20260827)
teaching_population = np.clip(rng.normal(47.3, 6.2, 600), 30, 70)

def repeated_means(population, sample_size, repeats=500, seed=1):
    local_rng = np.random.default_rng(seed)
    return np.array([local_rng.choice(population, size=sample_size, replace=False).mean()
                     for _ in range(repeats)])

means_10 = repeated_means(teaching_population, 10, seed=10)
means_50 = repeated_means(teaching_population, 50, seed=50)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True)
for ax, vals, size in zip(axes, [means_10, means_50], [10, 50]):
    ax.hist(vals, bins=24, color='#277DA1', alpha=0.85)
    ax.axvline(teaching_population.mean(), color='#E76F51', lw=2)
    ax.set_title(f'n={size}；均值SD={vals.std(ddof=1):.3f}')
    ax.set_xlabel('样本均值（分钟）')
axes[0].set_ylabel('重复次数')
plt.tight_layout()
plt.show()

## 3. 标准误、95% t置信区间与单样本双侧t检验

- 标准误：`SE=s/√n`；
- 95%区间：`x̄ ± t* × SE`；
- 原假设：`H0: μ=50`；备择假设：`H1: μ≠50`。

In [ ]:
mu0 = 50.0
alpha = 0.05
dfree = n - 1
se = sd / np.sqrt(n)
t_critical = stats.t.ppf(1 - alpha/2, dfree)
ci_low = mean - t_critical * se
ci_high = mean + t_critical * se
t_stat = (mean - mu0) / se
p_value = 2 * stats.t.sf(abs(t_stat), dfree)

result = pd.DataFrame({
    'n':[n], 'mean':[mean], 'sd':[sd], 'se':[se],
    'ci_low':[ci_low], 'ci_high':[ci_high],
    't':[t_stat], 'df':[dfree], 'p_two_sided':[p_value]
})
result.round(6)

主案例的总体均值区间为`[44.936, 49.064]`分钟，不包含50分钟；双侧`p≈0.0062`。若改用“相对基准节省多少分钟”表达，节省量区间为`[0.936, 5.064]`分钟。

In [ ]:
mean_diff = mean - mu0
diff_low, diff_high = ci_low - mu0, ci_high - mu0
fig, ax = plt.subplots(figsize=(7.2, 2.8))
ax.errorbar(mean_diff, 0, xerr=[[mean_diff-diff_low], [diff_high-mean_diff]],
            fmt='o', color='#277DA1', capsize=8, lw=2.5)
ax.axvline(0, color='#263238', lw=2, label='0：无差异')
ax.axvline(-4, color='#2A9D8F', ls='--', lw=2, label='-4：工程关注线')
ax.set_yticks([])
ax.set_xlabel('相对50分钟基准的均值差（分钟）')
ax.legend(frameon=False)
plt.show()

In [ ]:
x_grid = np.linspace(-4.5, 4.5, 900)
y_grid = stats.t.pdf(x_grid, dfree)
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.plot(x_grid, y_grid, color='#263238', lw=2)
ax.fill_between(x_grid, 0, y_grid, where=np.abs(x_grid) >= abs(t_stat),
                color='#D1495B', alpha=0.55)
ax.axvline(t_stat, color='#277DA1', ls='--', lw=2)
ax.axvline(-t_stat, color='#277DA1', ls='--', lw=2)
ax.set_xlabel('t统计量')
ax.set_ylabel('密度')
ax.set_title(f'双侧p值区域：p={p_value:.4f}')
plt.show()

## 4. 比较案例B

案例B只提供汇总统计量：`n=25`、`x̄=48`、`s=5`、`μ0=50`。请沿用主案例的计算顺序。

In [ ]:
case_b = pd.DataFrame({
    'n':[25], 'mean':[48.0], 'sd':[5.0], 'se':[1.0],
    'ci_low':[48.0 - t_critical], 'ci_high':[48.0 + t_critical],
    't':[-2.0], 'df':[24], 'p_two_sided':[2 * stats.t.sf(2.0, 24)],
    'engineering_threshold_min':[4.0]
})
case_b.round(6)

## 5. 输出判读模板

1. 样本均值与基准相差多少？
2. 95%区间是否包含50分钟？
3. `p`与`α=0.05`怎样比较，统计决定是什么？
4. 节省量区间是否确定超过4分钟工程关注线？

### 学生任务

请在下方填写案例B的四项判读，并在最后一项写150—200字完整结论。

In [ ]:
student_answer = {
    '样本差值_分钟': '',
    '区间是否包含50': '',
    'alpha_0.05下决定': '',
    '统计与工程解释_150至200字': '',
}
student_answer

## 6. 课后任务

保存运行后的Notebook；完成150—200字解释；按M06命名并提交。项目材料位于`project`文件夹。